# Assignment 1 — Build Your Own Simple Agent

**Goal:** an agent that takes a query, **decides** which tool to use, runs it, observes the result, and answers — looping through **Thought -> Action -> Observation -> Answer** (the *ReAct* pattern).

Two tools: `search_tool` (DuckDuckGo) and `calculator_tool` (math). The LLM decides which to call.

In [24]:
%pip install transformers langchain langchain-community langchain-huggingface ddgs huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# --- LOCAL MODEL: runs offline, no token needed ---
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

llm = HuggingFacePipeline(pipeline=pipeline(
    'text2text-generation', model='google/flan-t5-base', max_new_tokens=256))
print('LLM ready: local flan-t5-base')



LLM ready: local flan-t5-base


## Define the two tools

Same tools you used in `ZeroDependencyAgent.ipynb` / `LangChain agent.ipynb`: a DuckDuckGo search and a safe calculator (regex guard so `eval` can only see numbers and math symbols).

In [26]:
import re
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun

# --- Tool 1: web search (DuckDuckGo) ---
_ddg = DuckDuckGoSearchRun()
_ddg.api_wrapper.backend = 'html'  # real page snippets

def search_tool(query: str) -> str:
    """Search the web and return a short text snippet."""
    try:
        return _ddg.run(query)[:500]
    except Exception as e:
        return f'Search error: {e}'

# --- Tool 2: calculator (safe arithmetic only) ---
def calculator_tool(expr: str) -> str:
    """Evaluate a math expression like '4.4 * 0.05'."""
    if not re.fullmatch(r'[0-9\.\+\-\*\/\(\) ]+', expr):
        return 'Calculator error: invalid expression.'
    try:
        return str(eval(expr))
    except Exception as e:
        return f'Calculator error: {e}'

# Registry the agent will pick from
TOOLS = {'search': search_tool, 'calculator': calculator_tool}
print('Tools available:', list(TOOLS))

Tools available: ['search', 'calculator']


## The ReAct loop

We give the model a **system prompt** that teaches it the format. On each step the model writes a `Thought` and an `Action` (`search` or `calculator`) with an `Action Input`. We stop generation at `Observation:`, **we** run the chosen tool, paste the real result back as the `Observation`, and let the model continue — until it writes `Final Answer:`.

In [27]:
SYSTEM_PROMPT = '''You are a reasoning agent. Answer the question using the tools.
Available tools:
- search: look something up on the web. Action Input = search query.
- calculator: do arithmetic. Action Input = a math expression like 12 * 3.

Use EXACTLY this format, one step at a time:
Thought: <your reasoning>
Action: <search or calculator>
Action Input: <input for the tool>
Observation: <result will be filled in for you>
... (repeat Thought/Action/Action Input/Observation as needed) ...
Thought: I now know the answer
Final Answer: <the answer>

Question: {question}
'''

def run_agent(question, max_steps=5, verbose=True):
    transcript = SYSTEM_PROMPT.format(question=question)
    for step in range(max_steps):
        out = llm.invoke(transcript)          # model writes Thought + Action + Action Input
        transcript += out
        if verbose:
            print(out.strip())

        if 'Final Answer:' in out:
            return out.split('Final Answer:')[-1].strip()

        # parse the action the model chose
        action = re.search(r'Action:\s*(\w+)', out)
        ainput = re.search(r'Action Input:\s*(.+)', out)
        if not action or not ainput:
            return out.strip()                # model answered without a tool
        name, arg = action.group(1).strip(), ainput.group(1).strip()

        # ACTION: run the tool ; OBSERVATION: feed the result back
        result = TOOLS.get(name, lambda x: f'Unknown tool: {name}')(arg)
        observation = f'\nObservation: {result}\n'
        transcript += observation
        if verbose:
            print(observation.strip(), '\n')
    return 'Stopped: max steps reached.'

## Try it — a question that needs **both** tools (search, then calculate):

In [28]:
answer = run_agent('What is the GDP of Germany in 2023, and what is 5% of that number?')
print('\n=== FINAL ANSWER ===')
print(answer)

5%

=== FINAL ANSWER ===
5%


A calculator-only question:

In [29]:
print(run_agent('What is 1234 * 17?'))

calculator: do arithmetic. Action Input = a math expression like 12 * 3.
calculator: do arithmetic. Action Input = a math expression like 12 * 3.


summary

- **Thought** — the LLM reasons about what to do next.
- **Action / Action Input** — the LLM names a tool and its argument.
- **Observation** — *our code* runs the tool and feeds the real result back.
- **Final Answer** — the LLM stops looping and responds.
